## Cell 0 — Environment & Global Settings

In [1]:
# ============================================================
# CELL 0: Environment & Global Settings
# HASEM – Complete Submission-Ready Notebook
# ============================================================
# Installs
!pip install -q gprofiler-official mygene scikit-learn xgboost shap

# ── Core imports ─────────────────────────────────────────────
import os, sys, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f as f_dist, binom

# ── Bioinformatics ───────────────────────────────────────────
from gprofiler import GProfiler
import mygene

# ── Sklearn ──────────────────────────────────────────────────
from sklearn.impute          import SimpleImputer
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.decomposition   import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, IsolationForest
from sklearn.calibration     import calibration_curve, CalibratedClassifierCV
from sklearn.metrics         import (
    roc_auc_score, average_precision_score, accuracy_score,
    f1_score, recall_score, confusion_matrix, brier_score_loss,
    RocCurveDisplay, PrecisionRecallDisplay, classification_report
)

# ── Optional XGBoost / SHAP ──────────────────────────────────
try:
    import xgboost as xgb
    HAVE_XGB = True
except ImportError:
    HAVE_XGB = False

try:
    import shap
    HAVE_SHAP = True
except ImportError:
    HAVE_SHAP = False

# ── Reproducibility ──────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Processing constants ─────────────────────────────────────
CHUNK_SIZE          = 5000
TOP_K               = 5000      # MAD-selected genes
VAR_EXPLAINED       = 0.95      # PCA cumulative variance threshold
LOW_EXPR_THRESHOLD  = 1.0       # minimum median expression filter
ANOMALY_ALPHA       = 0.01      # threshold significance level
MIN_CANCER_SAMPLES  = 5         # minimum samples per cancer type

# ── Suppress noisy warnings ──────────────────────────────────
warnings.filterwarnings("ignore")
pd.set_option("future.no_silent_downcasting", True)

# ── Input paths ──────────────────────────────────────────────
EXPR_PATH  = "/kaggle/input/datasets/kaythie/cancer-dataset/TcgaTargetGtex_gene_expected_count/TcgaTargetGtex_gene_expected_count"
PHENO_PATH = "/kaggle/input/datasets/kaythie/cancer-dataset/TcgaTargetGTEX_phenotype.txt/TcgaTargetGTEX_phenotype.txt"

# ── Output directories ────────────────────────────────────────
OUTDIR = "results_hasem"
for sub in ["preprocessing","batch_effects","unsupervised","loco",
            "calibration","supervised","per_cancer","pathway_enrichment",
            "contributions","models","logs"]:
    os.makedirs(os.path.join(OUTDIR, sub), exist_ok=True)

# ── Healthy sample tags ───────────────────────────────────────
HEALTHY_TAGS = ["GTEX", "Normal Tissue", "Solid Tissue Normal"]

# ── Simple logger ─────────────────────────────────────────────
def log(msg):
    print(f"[HASEM] {msg}")

# ── Save config snapshot ──────────────────────────────────────
pd.Series({
    "RANDOM_SEED": RANDOM_SEED,
    "TOP_K": TOP_K,
    "VAR_EXPLAINED": VAR_EXPLAINED,
    "LOW_EXPR_THRESHOLD": LOW_EXPR_THRESHOLD,
    "ANOMALY_ALPHA": ANOMALY_ALPHA,
    "HAVE_XGB": HAVE_XGB,
    "HAVE_SHAP": HAVE_SHAP
}).to_csv(os.path.join(OUTDIR, "config_snapshot.csv"))

log("Environment ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 1.2 MB/s eta 0:00:0000:01
[HASEM] Environment ready.


## Cell 1 — Data Loading, MAD Feature Selection & Healthy-Baseline PCA

In [2]:
# ============================================================
# CELL 1: Data Loading, MAD Feature Selection & Healthy PCA
# ============================================================
log("Step 1a: Computing MAD scores across all genes...")

# ── Step 1a: Compute MAD in chunks ────────────────────────────
mad_scores = []
with pd.read_csv(
    EXPR_PATH, sep="\t", index_col=0, chunksize=CHUNK_SIZE,
    engine="c", dtype_backend="numpy_nullable"
) as reader:
    for chunk in reader:
        chunk_median = chunk.median(axis=1)
        chunk = chunk.loc[chunk_median >= LOW_EXPR_THRESHOLD]
        if chunk.empty:
            continue
        med = chunk.median(axis=1)
        mad = (chunk.sub(med, axis=0)).abs().median(axis=1)
        mad_scores.extend(zip(chunk.index, mad))

mad_df = pd.DataFrame(mad_scores, columns=["Gene","MAD"]).drop_duplicates("Gene")
mad_df["MAD_z"] = (mad_df["MAD"] - mad_df["MAD"].mean()) / mad_df["MAD"].std()
top_genes = mad_df.nlargest(TOP_K, "MAD_z")["Gene"].tolist()

log(f"Genes scanned: {len(mad_df):,} | Top {TOP_K} selected by MAD z-score")

# ── Step 1b: Reload top genes & align phenotype ───────────────
log("Step 1b: Reloading top genes and aligning phenotype data...")

chunks = []
with pd.read_csv(
    EXPR_PATH, sep="\t", index_col=0, chunksize=CHUNK_SIZE,
    engine="c", dtype_backend="numpy_nullable"
) as reader:
    for chunk in reader:
        overlap = chunk.index.intersection(top_genes)
        if not overlap.empty:
            chunks.append(chunk.loc[overlap])
expr_top = pd.concat(chunks).T
expr_top.index.name = "sample"

meta = pd.read_csv(PHENO_PATH, sep="\t", encoding="latin1", low_memory=False)
meta.rename(columns={"sample_type":"_sample_type","primary_site":"_primary_site","Sample":"sample"}, inplace=True)
meta = meta.set_index("sample", drop=False)

common = expr_top.index.intersection(meta["sample"])
expr_top = expr_top.loc[common]
meta     = meta.loc[common]
log(f"Aligned dataset: {expr_top.shape[0]:,} samples × {expr_top.shape[1]:,} genes")

# ── Step 1c: Labels ───────────────────────────────────────────
meta["is_tumor"]  = (~meta["_sample_type"].isin(HEALTHY_TAGS)).astype(int)
meta["cohort"]    = meta["_sample_type"].apply(
    lambda x: "GTEx" if x in HEALTHY_TAGS else "TCGA/TARGET")
y_binary   = meta["is_tumor"]
y_detailed = meta["_primary_site"].copy()
y_detailed[y_binary == 0] = "HEALTHY"

log(f"Healthy: {(y_binary==0).sum():,} | Tumour: {(y_binary==1).sum():,}")

# ── Step 1d: Preprocess (log2, impute, scale on healthy only) ─
expr_numeric = expr_top.apply(pd.to_numeric, errors="coerce")
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(expr_numeric),
    index=expr_numeric.index, columns=expr_numeric.columns)
X_log = np.log2(X_imputed.clip(lower=1e-6) + 1)

scaler = StandardScaler()
scaler.fit(X_log[y_binary == 0])            # ← ONLY on healthy
X_scaled = pd.DataFrame(
    scaler.transform(X_log),
    index=X_log.index, columns=X_log.columns)
X_scaled.replace([np.inf, -np.inf], np.nan, inplace=True)

[HASEM] Step 1a: Computing MAD scores across all genes...
[HASEM] Genes scanned: 26,370 | Top 5000 selected by MAD z-score
[HASEM] Step 1b: Reloading top genes and aligning phenotype data...
[HASEM] Aligned dataset: 19,109 samples × 5,000 genes
[HASEM] Healthy: 8,149 | Tumour: 10,960


## **Cell 1B — ComBat + PCA (the complete replacement)**

In [3]:
# ============================================================
# CELL 1B: ComBat Harmonisation + PCA on Healthy Baseline
# Must run AFTER Cell 1 and BEFORE Cell 2
# ============================================================
!pip install -q neuroCombat

from neuroCombat import neuroCombat

log("Step 1B-i: Running ComBat batch harmonisation...")

# ── ComBat expects (genes × samples) — transpose X_scaled ────
batch_labels = (meta["cohort"] == "TCGA/TARGET").astype(int).values

covars = pd.DataFrame({
    "batch":    batch_labels,
    "is_tumor": y_binary.values
})

combat_output = neuroCombat(
    dat              = X_scaled.values.T,
    covars           = covars,
    batch_col        = "batch",
    categorical_cols = ["is_tumor"],
    mean_only        = True       
)

# ── Replace X_scaled with harmonised version ──────────────────
X_scaled_original = X_scaled.copy()        # keep backup
X_scaled = pd.DataFrame(
    combat_output["data"].T,               # back to (samples × genes)
    index   = X_scaled.index,
    columns = X_scaled.columns
)
X_scaled.replace([np.inf, -np.inf], np.nan, inplace=True)
log(f"ComBat complete. Shape: {X_scaled.shape}")

# ── Sanity check: cohort means should now be similar ──────────
gtex_mean = X_scaled[meta["cohort"]=="GTEx"].mean().mean()
tcga_mean = X_scaled[meta["cohort"]=="TCGA/TARGET"].mean().mean()
log(f"Post-ComBat mean — GTEx: {gtex_mean:.4f} | TCGA/TARGET: {tcga_mean:.4f}")
log(f"Mean difference: {abs(gtex_mean-tcga_mean):.4f} (near 0 = good)")

# ── Step 1e (moved here): PCA on ComBat-corrected healthy ─────
log("Step 1B-ii: Fitting PCA on ComBat-corrected healthy samples...")
mask_healthy = (y_binary == 0).values

pca_unsup = PCA(n_components=VAR_EXPLAINED, svd_solver="full", random_state=RANDOM_SEED)
pca_unsup.fit(X_scaled.values[mask_healthy])   # ← now fitted on CLEAN data
X_pca = pca_unsup.transform(X_scaled.values)
n_comp = pca_unsup.n_components_
log(f"PCA components retained: {n_comp} (≥{VAR_EXPLAINED*100:.0f}% variance)")

# ── PCA variance plot ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(np.cumsum(pca_unsup.explained_variance_ratio_), color="#1F4E79", lw=1.5)
ax.axhline(VAR_EXPLAINED, color="red", ls="--", lw=1,
           label=f"{VAR_EXPLAINED*100:.0f}% threshold")
ax.set_xlabel("Principal Components")
ax.set_ylabel("Cumulative Variance Explained")
ax.set_title("PCA Scree – ComBat-Corrected Healthy Samples")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,"preprocessing","pca_scree_combat.png"), dpi=150)
plt.close()

# ── Save all artefacts ────────────────────────────────────────
meta.to_csv(os.path.join(OUTDIR,"models","meta_aligned.csv"))
joblib.dump(
    {"scaler":scaler, "pca_unsup":pca_unsup,
     "top_genes":top_genes, "combat_output":combat_output},
    os.path.join(OUTDIR,"models","preproc_artefacts.joblib")
)
log("Step 1B complete. All downstream cells will now use ComBat-corrected data.")

  Preparing metadata (setup.py) ... done
[HASEM] Step 1B-i: Running ComBat batch harmonisation...
[neuroCombat] Creating design matrix
[neuroCombat] Standardizing data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Final adjustment of data
[HASEM] ComBat complete. Shape: (19109, 5000)
[HASEM] Post-ComBat mean — GTEx: -0.0000 | TCGA/TARGET: 0.0889
[HASEM] Mean difference: 0.0889 (near 0 = good)
[HASEM] Step 1B-ii: Fitting PCA on ComBat-corrected healthy samples...
[HASEM] PCA components retained: 1609 (≥95% variance)
[HASEM] Step 1B complete. All downstream cells will now use ComBat-corrected data.


## Cell 2 — Batch-Effect Analysis

Before building the detector, we check whether cohort origin (GTEx vs TCGA/TARGET) drives the dominant structure in the data. If the first PCA components separate cohorts perfectly, the model may be learning *where the data came from* rather than *whether it is healthy or cancerous*.

In [4]:
# ============================================================
# CELL 2: Batch-Effect Analysis (FIXED — real pre-ComBat comparison)
# Three checks:
#   A. PCA biplot coloured by cohort (visual check)
#   B. Cohort separability test, BEFORE and AFTER ComBat
#   C. TCGA solid-normal hold-out check (are TCGA normals
#      correctly kept below the anomaly threshold?)
# ============================================================
log("Step 2: Batch-effect analysis...")
batch_dir = os.path.join(OUTDIR, "batch_effects")

# ── 2A: PCA biplot coloured by cohort ────────────────────────
X_pca_2d = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X_scaled.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cohort_labels = meta["cohort"].values

for ax, (title, mask_fn) in zip(axes, [
    ("Coloured by Cohort",   lambda: cohort_labels),
    ("Coloured by Tumour/Healthy", lambda: np.where(y_binary.values==0,"Healthy","Tumour"))
]):
    labels = mask_fn()
    unique = np.unique(labels)
    palette = {"GTEx":"#2196F3","TCGA/TARGET":"#F44336",
               "Healthy":"#4CAF50","Tumour":"#F44336"}
    for lab in unique:
        m = labels == lab
        ax.scatter(X_pca_2d[m,0], X_pca_2d[m,1],
                   s=3, alpha=0.4, label=lab,
                   color=palette.get(lab,"#888888"))
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title(title)
    ax.legend(markerscale=4, fontsize=9); ax.grid(alpha=0.2)

plt.suptitle("PCA Biplot — Batch vs Biological Signal", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(batch_dir,"pca_biplot_cohort_vs_tumour.png"), dpi=150)
plt.close()
log("2A: PCA biplot saved.")

# ── 2B: Cohort separability test — BEFORE and AFTER ComBat ───
def cohort_separability(X_df, label, seed=RANDOM_SEED):
    cohort_enc = LabelEncoder().fit_transform(meta["cohort"].values)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_df.values, cohort_enc, test_size=0.2,
        stratify=cohort_enc, random_state=seed)
    lr_batch = LogisticRegression(max_iter=1000, solver="saga", random_state=seed)
    lr_batch.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, lr_batch.predict_proba(X_te)[:, 1])
    log(f"    {label}: cohort-separability ROC-AUC = {auc:.4f}")
    return auc

log("2B: Measuring cohort separability before and after ComBat...")
pre_combat_auc = cohort_separability(X_scaled_original, "Pre-ComBat")
post_combat_auc = cohort_separability(X_scaled, "Post-ComBat")
batch_auc = post_combat_auc  # keep downstream variable name intact

log(f"    (AUC near 0.5 = no batch; AUC near 1.0 = strong batch signal)")
if post_combat_auc > 0.80:
    log("    WARNING: Strong cohort separability remains after ComBat. "
        "Report this honestly — mean-only correction may be insufficient.")
else:
    log("    OK: ComBat meaningfully reduced cohort separability.")

# ── 2C: TCGA solid-normal hold-out check ─────────────────────
tcga_normal_mask = meta["_sample_type"] == "Solid Tissue Normal"
n_tcga_normals = tcga_normal_mask.sum()
log(f"2C: Found {n_tcga_normals} TCGA Solid-Tissue-Normal samples for hold-out check.")

batch_summary = pd.DataFrame({
    "Check": ["Cohort Separability ROC-AUC (pre-ComBat)",
              "Cohort Separability ROC-AUC (post-ComBat)",
              "Improvement",
              "TCGA Solid-Normal samples available",
              "Interpretation"],
    "Result": [f"{pre_combat_auc:.4f}",
               f"{post_combat_auc:.4f}",
               f"{pre_combat_auc - post_combat_auc:.4f}",
               str(n_tcga_normals),
               "Low batch effect" if post_combat_auc < 0.80 else "Batch effect present — review recommended"]
})
batch_summary.to_csv(os.path.join(batch_dir,"batch_effect_summary.csv"), index=False)
log("Step 2 (Batch-Effect Analysis) complete.")

[HASEM] Step 2: Batch-effect analysis...
[HASEM] 2A: PCA biplot saved.
[HASEM] 2B: Measuring cohort separability before and after ComBat...
[HASEM]     Pre-ComBat: cohort-separability ROC-AUC = 0.9998
[HASEM]     Post-ComBat: cohort-separability ROC-AUC = 0.9998
[HASEM]     (AUC near 0.5 = no batch; AUC near 1.0 = strong batch signal)
[HASEM]     WARNING: Strong cohort separability remains after ComBat. Report this honestly — mean-only correction may be insufficient.
[HASEM] 2C: Found 737 TCGA Solid-Tissue-Normal samples for hold-out check.
[HASEM] Step 2 (Batch-Effect Analysis) complete.


In [5]:
# ── Genuine batch test: GTEx vs TCGA-normal, healthy samples only ──
healthy_only = (y_binary.values == 0)
source_label = (meta["_sample_type"].values[healthy_only] == "Solid Tissue Normal").astype(int)  # 1=TCGA-normal, 0=GTEx
log(f"Healthy-only source split: {(source_label==0).sum()} GTEx, {(source_label==1).sum()} TCGA-normal")

Xh_tr, Xh_te, sh_tr, sh_te = train_test_split(
    X_scaled.values[healthy_only], source_label, test_size=0.2,
    stratify=source_label, random_state=RANDOM_SEED)
lr_src = LogisticRegression(max_iter=1000, solver="saga", random_state=RANDOM_SEED)
lr_src.fit(Xh_tr, sh_tr)
source_auc = roc_auc_score(sh_te, lr_src.predict_proba(Xh_te)[:, 1])
log(f"GTEx-vs-TCGA-normal separability (within healthy class): ROC-AUC = {source_auc:.4f}")

pd.DataFrame([{"Check": "GTEx vs TCGA-normal (within healthy, true batch test)", "ROC_AUC": source_auc}]
    ).to_csv(os.path.join(batch_dir, "true_batch_negative_control.csv"), index=False)

[HASEM] Healthy-only source split: 7412 GTEx, 737 TCGA-normal
[HASEM] GTEx-vs-TCGA-normal separability (within healthy class): ROC-AUC = 1.0000


## Cell 3 — MSPC Anomaly Scoring: Hotelling T² and Q-Residual

In [6]:
# ============================================================
# CELL 3: MSPC Anomaly Scoring (T², Q, thresholds, flags)
# ============================================================
log("Step 3: Computing T² and Q-residual anomaly scores...")

X_arr = X_scaled.values
X_pca_arr  = pca_unsup.transform(X_arr)
X_recon    = pca_unsup.inverse_transform(X_pca_arr)
residuals  = X_arr - X_recon

safe_eigen = np.where(pca_unsup.explained_variance_ <= 0,
                      1e-12, pca_unsup.explained_variance_)
T2_scores  = np.sum((X_pca_arr ** 2) / safe_eigen, axis=1)
Q_scores   = np.sum(residuals ** 2, axis=1)

m = mask_healthy.sum()   # number of healthy samples
a = n_comp               # number of PC components

# ── Empirical thresholds (99th percentile on healthy) ────────
T2_thresh_emp = np.nanpercentile(T2_scores[mask_healthy], 99)
Q_thresh_emp  = np.nanpercentile(Q_scores[mask_healthy],  99)

# ── Theoretical T² threshold (Hotelling F-distribution) ──────
if m > a:
    Fcrit = f_dist.ppf(1 - ANOMALY_ALPHA, a, m - a)
    T2_thresh_theo = (a * (m-1) * (m+1)) / (m * (m-a)) * Fcrit
else:
    T2_thresh_theo = T2_thresh_emp
    log("Warning: insufficient healthy samples for theoretical T² — using empirical.")

# ── Theoretical Q threshold (Jackson–Mudholkar) ───────────────
try:
    resid_var = np.var(residuals[mask_healthy], axis=0)
    th1 = resid_var.sum()
    th2 = (resid_var**2).sum()
    th3 = (resid_var**3).sum()
    h0  = 1 - (2*th1*th3) / (3*th2**2 + 1e-12)
    if h0 <= 0 or th2 <= 0:
        raise ValueError("Unstable Q parameters")
    ca  = np.sqrt(2*th2*(h0**2)) / (th1 + 1e-12)
    fa  = f_dist.ppf(1-ANOMALY_ALPHA, max(1,round(h0*2)), max(1,round(h0*2)))
    Q_thresh_theo = th1 * ((np.sqrt(fa)*ca + 1 +
                            th2*h0*(h0-1)/th1**2) ** (1.0/h0))
except Exception as e:
    Q_thresh_theo = Q_thresh_emp
    log(f"Q-theoretical fallback to empirical: {e}")

# ── Binary anomaly flags ──────────────────────────────────────
y_pred_emp  = ((T2_scores > T2_thresh_emp)  | (Q_scores > Q_thresh_emp )).astype(int)
y_pred_theo = ((T2_scores > T2_thresh_theo) | (Q_scores > Q_thresh_theo)).astype(int)

# ── Combined continuous score (z-normalised) ─────────────────
T2_z  = (T2_scores - T2_scores[mask_healthy].mean()) / (T2_scores[mask_healthy].std() + 1e-12)
Q_z   = (Q_scores  - Q_scores[mask_healthy].mean())  / (Q_scores[mask_healthy].std()  + 1e-12)
combined_score = (T2_z + Q_z) / 2.0

# ── Isolation Forest benchmark ────────────────────────────────
iso = IsolationForest(random_state=RANDOM_SEED, contamination="auto", n_jobs=-1)
iso.fit(X_arr[mask_healthy])
iso_scores = -iso.decision_function(X_arr)
iso_preds  = (iso.predict(X_arr) == -1).astype(int)

# ── Performance helper ────────────────────────────────────────
def evaluate(y_true, y_pred, score, label):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    roc = roc_auc_score(y_true, score)
    pr  = average_precision_score(y_true, score)
    det = 100 * tp / (tp + fn + 1e-12)
    fal = 100 * fp / (fp + tn + 1e-12)
    print(f"  {label:30s}  ROC={roc:.4f}  PR={pr:.4f}  "
          f"Det={det:.2f}%  FA={fal:.2f}%")
    return dict(label=label, roc=roc, pr=pr, detect=det, false_alarm=fal,
                tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))

print("\n[HASEM] Unsupervised detection results:")
res_emp  = evaluate(y_binary.values, y_pred_emp,  combined_score, "T²+Q Empirical")
res_theo = evaluate(y_binary.values, y_pred_theo, combined_score, "T²+Q Theoretical")
res_iso  = evaluate(y_binary.values, iso_preds,   iso_scores,     "Isolation Forest")

metrics_df = pd.DataFrame([res_emp, res_theo, res_iso])
metrics_df.to_csv(os.path.join(OUTDIR,"unsupervised","unsup_metrics.csv"), index=False)

# ── T² vs Q scatter plot ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8,6))
colors_map = {0:"#4CAF50", 1:"#F44336"}
for label, grp in pd.DataFrame({"T2":T2_scores,"Q":Q_scores,"tumor":y_binary.values}).groupby("tumor"):
    ax.scatter(grp["T2"], grp["Q"], s=4, alpha=0.3,
               color=colors_map[label],
               label="Healthy" if label==0 else "Tumour")
ax.axvline(T2_thresh_emp, color="#1F4E79", ls="--", lw=1.2, label=f"T² threshold")
ax.axhline(Q_thresh_emp,  color="#FF6600", ls="--", lw=1.2, label=f"Q threshold")
ax.set_xlabel("Hotelling T²"); ax.set_ylabel("Q-Residual")
ax.set_title("MSPC Anomaly Map: T² vs Q"); ax.legend(markerscale=4)
ax.set_xlim(left=0); ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,"unsupervised","T2_Q_scatter.png"), dpi=150)
plt.close()
log("Step 3 complete.")

# ── TCGA Solid-Normal holdout evaluation (add at end of Cell 3) ──
tcga_normal_mask = (meta["_sample_type"] == "Solid Tissue Normal").values
n_tn = tcga_normal_mask.sum()
if n_tn > 0:
    fa_tcga = 100 * y_pred_emp[tcga_normal_mask].mean()
    log(f"TCGA Solid-Normal holdout false alarm rate: {fa_tcga:.2f}% (n={n_tn})")
    log(f"  (This is your batch-effect defence — should be low)")


[HASEM] Step 3: Computing T² and Q-residual anomaly scores...

[HASEM] Unsupervised detection results:
  T²+Q Empirical                  ROC=0.9920  PR=0.9950  Det=97.86%  FA=2.01%
  T²+Q Theoretical                ROC=0.9920  PR=0.9950  Det=85.61%  FA=13.34%
  Isolation Forest                ROC=0.8824  PR=0.8501  Det=59.81%  FA=10.21%
[HASEM] Step 3 complete.
[HASEM] TCGA Solid-Normal holdout false alarm rate: 0.81% (n=737)
[HASEM]   (This is your batch-effect defence — should be low)


In [7]:
# ── Does the source signal leak into false alarms? ──
gtex_mask = healthy_only.copy()
gtex_mask[healthy_only] = (source_label == 0)   # GTEx healthy samples, in full-array positions
tcga_norm_mask = healthy_only.copy()
tcga_norm_mask[healthy_only] = (source_label == 1)  # TCGA-normal healthy samples

pred_gtex = ((T2_scores[gtex_mask] > T2_thresh_emp) | (Q_scores[gtex_mask] > Q_thresh_emp))
pred_tcga_norm = ((T2_scores[tcga_norm_mask] > T2_thresh_emp) | (Q_scores[tcga_norm_mask] > Q_thresh_emp))

fa_gtex = 100 * pred_gtex.mean()
fa_tcga_norm = 100 * pred_tcga_norm.mean()
log(f"False-alarm rate, GTEx healthy (n={gtex_mask.sum()}): {fa_gtex:.2f}%")
log(f"False-alarm rate, TCGA-normal healthy (n={tcga_norm_mask.sum()}): {fa_tcga_norm:.2f}%")

pd.DataFrame([
    {"Group": "GTEx healthy", "N": int(gtex_mask.sum()), "FalseAlarmRate": fa_gtex},
    {"Group": "TCGA-normal healthy", "N": int(tcga_norm_mask.sum()), "FalseAlarmRate": fa_tcga_norm}
]).to_csv(os.path.join(batch_dir, "false_alarm_by_source.csv"), index=False)

[HASEM] False-alarm rate, GTEx healthy (n=7412): 2.13%
[HASEM] False-alarm rate, TCGA-normal healthy (n=737): 0.81%


## Cell 4 — Full Per-Cancer Detection Rates

This cell produces the complete per-cancer sensitivity table required by reviewers. It reports detection rate, sample size, and 95% confidence interval for every cancer type in the dataset.

In [8]:
# ============================================================
# CELL 4: Full Per-Cancer Detection Rates
# Produces a complete table with:
#   - Cancer type, sample size
#   - Detection rate (%)
#   - 95% confidence interval (Wilson score interval)
# ============================================================
log("Step 4: Computing full per-cancer detection rates...")

def wilson_ci(k, n, z=1.96):
    """Wilson score 95% confidence interval for a proportion."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    margin = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return round((centre - margin)*100, 2), round((centre + margin)*100, 2)

def full_per_cancer_table(y_true_arr, y_pred_arr, cancer_labels, tag=""):
    rows = []
    mask_tumor = (y_true_arr == 1)
    df = pd.DataFrame({
        "cancer": cancer_labels[mask_tumor],
        "detected": y_pred_arr[mask_tumor]
    })
    for cancer, grp in df.groupby("cancer"):
        n   = len(grp)
        k   = grp["detected"].sum()
        lo, hi = wilson_ci(k, n)
        rows.append({
            "Cancer Type": cancer,
            "N Samples": n,
            "Detected": int(k),
            "Detection Rate (%)": round(100*k/n, 2) if n > 0 else np.nan,
            "95% CI Lower": lo,
            "95% CI Upper": hi
        })
    result = pd.DataFrame(rows).sort_values("Detection Rate (%)", ascending=False)
    result.to_csv(
        os.path.join(OUTDIR, "per_cancer", f"per_cancer_{tag}.csv"), index=False)
    return result

pc_emp  = full_per_cancer_table(y_binary.values, y_pred_emp,
                                y_detailed.values, "empirical")
pc_theo = full_per_cancer_table(y_binary.values, y_pred_theo,
                                y_detailed.values, "theoretical")

print("\n[HASEM] Per-cancer detection rates (Empirical threshold) — all types:")
print(pc_emp.to_string(index=False))

# ── Horizontal bar chart ──────────────────────────────────────
plot_df = pc_emp[pc_emp["Cancer Type"] != "HEALTHY"].copy()
fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df)*0.35)))
colors = ["#1F4E79" if r >= 90 else "#2196F3" if r >= 70
          else "#FF9800" if r >= 50 else "#F44336"
          for r in plot_df["Detection Rate (%)"]]
bars = ax.barh(plot_df["Cancer Type"], plot_df["Detection Rate (%)"],
               color=colors, edgecolor="white", height=0.7)
# Error bars (confidence intervals)
ax.errorbar(
    plot_df["Detection Rate (%)"],
    range(len(plot_df)),
    xerr=[plot_df["Detection Rate (%)"] - plot_df["95% CI Lower"],
          plot_df["95% CI Upper"]    - plot_df["Detection Rate (%)"]],
    fmt="none", color="black", capsize=3, lw=1
)
ax.axvline(90, color="green", ls="--", lw=1, label="90% line")
ax.set_xlabel("Detection Rate (%)"); ax.set_xlim(0, 105)
ax.set_title("Per-Cancer Detection Rates with 95% CI\n(HASEM T²+Q Empirical Threshold)")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,"per_cancer","per_cancer_detection_rates.png"),
            dpi=150, bbox_inches="tight")
plt.close()

log(f"Step 4 complete. Results for {len(pc_emp)} cancer types saved.")


[HASEM] Step 4: Computing full per-cancer detection rates...

[HASEM] Per-cancer detection rates (Empirical threshold) — all types:
               Cancer Type  N Samples  Detected  Detection Rate (%)  95% CI Lower  95% CI Upper
             Adrenal gland         77        77              100.00         95.25        100.00
                 Bile duct         36        36              100.00         90.36        100.00
                     Blood        107       107              100.00         96.53        100.00
               Bone Marrow         70        70              100.00         94.80        100.00
                    Cervix        305       305              100.00         98.76        100.00
                       Eye         79        79              100.00         95.36        100.00
               Endometrium        181       181              100.00         97.92        100.00
                     Colon        289       289              100.00         98.69        100.00
    

In [9]:
# ============================================================
# CELL 4B: Cell-Line Sensitivity Check
# The tumour class currently includes 433 "Cell Line" samples,
# which are lab-grown and technically distinct from patient
# tissue. This cell recomputes the headline unsupervised and
# per-cancer numbers with them excluded, and reports how the
# cell lines themselves score as a separate sanity check.
# ============================================================
log("Step 4B: Cell-line sensitivity check...")

cellline_mask = (meta["_sample_type"] == "Cell Line").values
log(f"Found {cellline_mask.sum()} cell-line samples out of {len(meta)} total.")

# Patient-only mask: all healthy samples + tumour samples that are NOT cell lines
patient_mask = (~cellline_mask)
n_removed = cellline_mask.sum()
log(f"Patient-only analysis set: {patient_mask.sum()} samples "
    f"({mask_healthy.sum()} healthy + {(y_binary.values[patient_mask]==1).sum()} patient tumours)")

# ── Recompute unsupervised metrics on patient-only data ──────
# Thresholds (T2_thresh_emp, T2_thresh_theo, Q_thresh_emp, Q_thresh_theo)
# and pca_unsup are unaffected — they were fit on healthy samples only,
# and no healthy sample is a cell line. Only the evaluation set changes.
T2_patient = T2_scores[patient_mask]
Q_patient  = Q_scores[patient_mask]
y_patient  = y_binary.values[patient_mask]
combined_patient = combined_score[patient_mask] if 'combined_score' in globals() else (
    (T2_patient - T2_patient[y_patient==0].mean())/T2_patient[y_patient==0].std()
    + (Q_patient - Q_patient[y_patient==0].mean())/Q_patient[y_patient==0].std())/2

pred_emp_patient  = ((T2_patient > T2_thresh_emp) | (Q_patient > Q_thresh_emp)).astype(int)
pred_theo_patient = ((T2_patient > T2_thresh_theo) | (Q_patient > Q_thresh_theo)).astype(int)

def quick_metrics(y_true, y_pred, score):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    roc = roc_auc_score(y_true, score)
    pr  = average_precision_score(y_true, score)
    det = tp/(tp+fn)*100
    fa  = fp/(tn+fp)*100
    return roc, pr, det, fa

roc_e, pr_e, det_e, fa_e = quick_metrics(y_patient, pred_emp_patient, combined_patient)
roc_t, pr_t, det_t, fa_t = quick_metrics(y_patient, pred_theo_patient, combined_patient)

log(f"  Empirical  (patient-only): ROC={roc_e:.4f} PR={pr_e:.4f} Det={det_e:.2f}% FA={fa_e:.2f}%")
log(f"  Theoretical(patient-only): ROC={roc_t:.4f} PR={pr_t:.4f} Det={det_t:.2f}% FA={fa_t:.2f}%")

comparison = pd.DataFrame({
    "Metric": ["ROC-AUC", "PR-AUC", "Detection %", "False Alarm %"],
    "All tumour samples (incl. cell lines)": [
        unsup_metrics_df.loc[unsup_metrics_df.label=="T²+Q Empirical","roc"].values[0]
        if 'unsup_metrics_df' in globals() else np.nan,
        np.nan, np.nan, np.nan],  # fill from your existing unsup_metrics.csv if desired
    "Patient tumours only (cell lines excluded)": [roc_e, pr_e, det_e, fa_e]
})
comparison.to_csv(os.path.join(OUTDIR, "unsupervised", "celline_sensitivity_comparison.csv"), index=False)

# ── How do the cell lines themselves score? ──────────────────
T2_cl = T2_scores[cellline_mask]
Q_cl  = Q_scores[cellline_mask]
pred_cl = ((T2_cl > T2_thresh_emp) | (Q_cl > Q_thresh_emp)).astype(int)
log(f"  Cell lines flagged as anomalous: {pred_cl.sum()}/{len(pred_cl)} "
    f"({100*pred_cl.mean():.2f}%)")

# ── Recompute per-cancer table, patient tumours only ──────────
y_detailed_patient = y_detailed.values[patient_mask]
df_percancer_patient = full_per_cancer_table(
    y_patient, pred_emp_patient, pd.Series(y_detailed_patient), tag="patientonly")

log("Step 4B complete. Saved celline_sensitivity_comparison.csv and per_cancer_empirical_patientonly.csv.")

[HASEM] Step 4B: Cell-line sensitivity check...
[HASEM] Found 433 cell-line samples out of 19109 total.
[HASEM] Patient-only analysis set: 18676 samples (8149 healthy + 10527 patient tumours)
[HASEM]   Empirical  (patient-only): ROC=0.9917 PR=0.9946 Det=97.78% FA=2.01%
[HASEM]   Theoretical(patient-only): ROC=0.9917 PR=0.9946 Det=85.02% FA=13.34%
[HASEM]   Cell lines flagged as anomalous: 433/433 (100.00%)
[HASEM] Step 4B complete. Saved celline_sensitivity_comparison.csv and per_cancer_empirical_patientonly.csv.


In [10]:
# ── T²-only vs Q-only ablation (uses variables already in memory) ──
for name, score, thresh in [("T² only", T2_scores, T2_thresh_emp), ("Q only", Q_scores, Q_thresh_emp)]:
    pred = (score > thresh).astype(int)
    roc, pr, det, fa = quick_metrics(y_binary.values, pred, score)
    log(f"  {name}: ROC={roc:.4f} PR={pr:.4f} Det={det:.2f}% FA={fa:.2f}%")
    pd.DataFrame([{"label": name, "roc": roc, "pr": pr, "detect": det, "false_alarm": fa}]).to_csv(
        os.path.join(OUTDIR, "unsupervised", f"ablation_{name.split()[0]}.csv"), index=False)

[HASEM]   T² only: ROC=0.8840 PR=0.9093 Det=24.95% FA=1.01%
[HASEM]   Q only: ROC=0.9978 PR=0.9986 Det=97.86% FA=1.01%


In [11]:
# ── AND rule: flag only if BOTH T² and Q exceed threshold ──
pred_and = ((T2_scores > T2_thresh_emp) & (Q_scores > Q_thresh_emp)).astype(int)
score_and = np.minimum(T2_scores / T2_thresh_emp, Q_scores / Q_thresh_emp)  # for ROC/PR ranking
roc, pr, det, fa = quick_metrics(y_binary.values, pred_and, score_and)
log(f"  T² AND Q: ROC={roc:.4f} PR={pr:.4f} Det={det:.2f}% FA={fa:.2f}%")
pd.DataFrame([{"label": "T² AND Q", "roc": roc, "pr": pr, "detect": det, "false_alarm": fa}]).to_csv(
    os.path.join(OUTDIR, "unsupervised", "ablation_AND.csv"), index=False)

[HASEM]   T² AND Q: ROC=0.8894 PR=0.9208 Det=24.95% FA=0.00%


## Cell 5 — LOCO Validation (External Validation Proxy)

**LOCO-A** uses the global PCA model to evaluate each cancer type individually.
**LOCO-B** re-trains PCA from scratch for each cancer type withheld — simulating what happens when a cancer type was *never seen during model building*. This is the closest available proxy to external validation.

In [12]:
# ============================================================
# CELL 5: Leave-One-Cancer-Out (LOCO) Validation
# LOCO-A: global model, per-cancer evaluation
# LOCO-B: re-trained PCA per cancer type (external validation proxy)
# ============================================================
log("Step 5: LOCO validation...")
loco_dir = os.path.join(OUTDIR, "loco")

# ── T² helper ────────────────────────────────────────────────
def compute_T2(Xpca, eigenvals):
    safe = np.where(eigenvals <= 0, 1e-12, eigenvals)
    return np.sum((Xpca**2) / safe, axis=1)

# ── LOCO-A ────────────────────────────────────────────────────
log("Running LOCO-A (global PCA, per-cancer evaluation)...")
loco_a_rows = []
for cancer in sorted(y_detailed[y_binary==1].dropna().unique()):
    mask = (y_detailed == cancer) & (y_binary == 1)
    n = mask.sum()
    if n < MIN_CANCER_SAMPLES:
        continue
    Xc     = X_arr[mask.values]
    Xc_pca = pca_unsup.transform(Xc)
    T2c    = compute_T2(Xc_pca, pca_unsup.explained_variance_)
    Qc     = np.sum((Xc - pca_unsup.inverse_transform(Xc_pca))**2, axis=1)
    det    = int(((T2c > T2_thresh_emp) | (Qc > Q_thresh_emp)).sum())
    lo, hi = wilson_ci(det, n)
    loco_a_rows.append(dict(
        Cancer=cancer, N=n, Detected=det,
        DetectRate=round(100*det/n,2),
        CI_Lower=lo, CI_Upper=hi
    ))

df_locoA = pd.DataFrame(loco_a_rows).sort_values("DetectRate", ascending=False)
df_locoA.to_csv(os.path.join(loco_dir,"LOCO_A_results.csv"), index=False)
log(f"LOCO-A complete: {len(df_locoA)} cancer types evaluated.")

# ── LOCO-B ────────────────────────────────────────────────────
log("Running LOCO-B (per-cancer retrained PCA — external validation proxy)...")
loco_b_rows = []
for cancer in sorted(y_detailed[y_binary==1].dropna().unique()):
    test_mask = (y_detailed == cancer) & (y_binary == 1)
    n_test    = test_mask.sum()
    if n_test < MIN_CANCER_SAMPLES:
        continue
    ref_mask  = (y_binary == 0)           # only healthy samples as reference
    if ref_mask.sum() < 30:
        continue

    # Standardise on reference (healthy) only
    Xref     = X_log.loc[ref_mask].values
    ref_mean = Xref.mean(0); ref_std = Xref.std(0, ddof=1)
    ref_std[ref_std == 0] = 1
    Xref_std = (Xref - ref_mean) / ref_std

    # Fit PCA on healthy only (this cancer type was never used)
    pca_tmp = PCA(n_components=VAR_EXPLAINED, svd_solver="full", random_state=RANDOM_SEED)
    pca_tmp.fit(Xref_std)
    eig = pca_tmp.explained_variance_

    # Theoretical T² threshold for this fold
    m_ref   = Xref_std.shape[0]
    nc      = pca_tmp.n_components_
    Fcrit   = f_dist.ppf(1-ANOMALY_ALPHA, nc, max(m_ref-nc,1))
    T2_lim  = (nc*(m_ref-1)*(m_ref+1)) / (m_ref*max(m_ref-nc,1)) * Fcrit
    Q_lim   = np.nanpercentile(
        np.sum((Xref_std - pca_tmp.inverse_transform(pca_tmp.transform(Xref_std)))**2, axis=1), 99)

    # Evaluate on withheld cancer type
    Xtest     = X_log.loc[test_mask].values
    Xtest_std = (Xtest - ref_mean) / ref_std
    Xtest_pca = pca_tmp.transform(Xtest_std)
    T2_test   = compute_T2(Xtest_pca, eig)
    Q_test    = np.sum((Xtest_std - pca_tmp.inverse_transform(Xtest_pca))**2, axis=1)
    det = int(((T2_test > T2_lim) | (Q_test > Q_lim)).sum())
    lo, hi = wilson_ci(det, n_test)
    loco_b_rows.append(dict(
        Cancer=cancer, N=n_test, Detected=det,
        DetectRate=round(100*det/n_test,2),
        CI_Lower=lo, CI_Upper=hi
    ))

df_locoB = pd.DataFrame(loco_b_rows).sort_values("DetectRate", ascending=False)
df_locoB.to_csv(os.path.join(loco_dir,"LOCO_B_results.csv"), index=False)
log(f"LOCO-B complete: {len(df_locoB)} cancer types evaluated.")

# ── Combined LOCO comparison plot ────────────────────────────
merged = df_locoA.merge(df_locoB, on="Cancer", suffixes=("_A","_B"))
fig, ax = plt.subplots(figsize=(10, max(6, len(merged)*0.35)))
y_pos = range(len(merged))
ax.barh([p+0.2 for p in y_pos], merged["DetectRate_A"],
        height=0.35, color="#1F4E79", label="LOCO-A (Global PCA)")
ax.barh([p-0.2 for p in y_pos], merged["DetectRate_B"],
        height=0.35, color="#2196F3", label="LOCO-B (Retrained PCA)")
ax.set_yticks(list(y_pos)); ax.set_yticklabels(merged["Cancer"], fontsize=8)
ax.set_xlabel("Detection Rate (%)"); ax.set_xlim(0,110)
ax.set_title("LOCO Validation: LOCO-A vs LOCO-B per Cancer Type")
ax.axvline(90, color="red", ls="--", lw=1, label="90% reference")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(loco_dir,"LOCO_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.close()
log("Step 5 complete.")


[HASEM] Step 5: LOCO validation...
[HASEM] Running LOCO-A (global PCA, per-cancer evaluation)...
[HASEM] LOCO-A complete: 32 cancer types evaluated.
[HASEM] Running LOCO-B (per-cancer retrained PCA — external validation proxy)...
[HASEM] LOCO-B complete: 32 cancer types evaluated.
[HASEM] Step 5 complete.


In [13]:
# ============================================================
# CELL 5C: True LOCO — organ never seen, healthy or diseased
# Removes BOTH the tumour samples of the held-out cancer type
# AND the matching-tissue healthy samples from the reference,
# so the model has genuinely never seen that organ at all.
# ============================================================
log("Step 5C: True LOCO (tissue-matched exclusion)...")

# meta['_primary_site'] holds tissue for every sample, healthy included —
# y_detailed overwrites this to 'HEALTHY' for non-tumour rows, so we
# read tissue straight from meta instead.
tissue_all = meta["_primary_site"].values
healthy_mask_arr = mask_healthy  # from Cell 3, boolean array over all samples

results = []
skipped = []

for cancer in sorted(y_detailed[y_binary == 1].dropna().unique()):
    test_mask = ((y_detailed == cancer) & (y_binary == 1)).values
    n_test = test_mask.sum()
    if n_test < MIN_CANCER_SAMPLES:
        continue

    # Which healthy samples share this tumour's tissue label?
    matching_healthy = healthy_mask_arr & (tissue_all == cancer)
    n_matching = matching_healthy.sum()

    # Reference = healthy samples EXCLUDING that tissue
    ref_mask = healthy_mask_arr & (tissue_all != cancer)
    n_ref = ref_mask.sum()

    if n_matching == 0:
        skipped.append((cancer, "no matching healthy tissue found — LOCO-B result applies unchanged"))
        continue
    if n_ref < 30:
        skipped.append((cancer, f"only {n_ref} healthy samples remain after exclusion — too few to refit"))
        continue

    Xref = X_log.values[ref_mask]
    ref_mean = Xref.mean(0)
    ref_std  = Xref.std(0, ddof=1)
    ref_std[ref_std == 0] = 1
    Xref_std = (Xref - ref_mean) / ref_std

    pca_tmp = PCA(n_components=VAR_EXPLAINED, svd_solver="full", random_state=RANDOM_SEED)
    pca_tmp.fit(Xref_std)
    eig = pca_tmp.explained_variance_
    nc  = pca_tmp.n_components_

    m_ref  = Xref_std.shape[0]
    Fcrit  = f_dist.ppf(1 - ANOMALY_ALPHA, nc, max(m_ref - nc, 1))
    T2_lim = (nc * (m_ref - 1) * (m_ref + 1)) / (m_ref * max(m_ref - nc, 1)) * Fcrit
    Q_ref  = np.sum((Xref_std - pca_tmp.inverse_transform(pca_tmp.transform(Xref_std)))**2, axis=1)
    Q_lim  = np.nanpercentile(Q_ref, 99)

    Xtest     = X_log.values[test_mask]
    Xtest_std = (Xtest - ref_mean) / ref_std
    Xtest_pca = pca_tmp.transform(Xtest_std)
    T2_test = compute_T2(Xtest_pca, eig)
    Q_test  = np.sum((Xtest_std - pca_tmp.inverse_transform(Xtest_pca))**2, axis=1)

    det = int(((T2_test > T2_lim) | (Q_test > Q_lim)).sum())
    lo, hi = wilson_ci(det, n_test)

    results.append(dict(
        Cancer=cancer, N_test=n_test, N_healthy_removed=n_matching,
        N_reference_remaining=n_ref, Detected=det,
        DetectRate=round(100*det/n_test, 2), CI_Lower=lo, CI_Upper=hi
    ))

df_true_loco = pd.DataFrame(results).sort_values("DetectRate", ascending=False)
df_true_loco.to_csv(os.path.join(OUTDIR, "loco", "LOCO_C_tissue_excluded.csv"), index=False)

log(f"Step 5C complete: {len(df_true_loco)} cancer types tested with tissue fully removed.")
if skipped:
    log(f"  {len(skipped)} cancer types skipped or unchanged:")
    for c, reason in skipped:
        log(f"    {c}: {reason}")

[HASEM] Step 5C: True LOCO (tissue-matched exclusion)...
[HASEM] Step 5C complete: 25 cancer types tested with tissue fully removed.
[HASEM]   7 cancer types skipped or unchanged:
[HASEM]     Adrenal gland: no matching healthy tissue found — LOCO-B result applies unchanged
[HASEM]     Bone Marrow: no matching healthy tissue found — LOCO-B result applies unchanged
[HASEM]     Eye: no matching healthy tissue found — LOCO-B result applies unchanged
[HASEM]     Lining of body cavities: no matching healthy tissue found — LOCO-B result applies unchanged
[HASEM]     Lymphatic tissue: no matching healthy tissue found — LOCO-B result applies unchanged
[HASEM]     SympatheticÊNervous System: no matching healthy tissue found — LOCO-B result applies unchanged
[HASEM]     White blood cell: no matching healthy tissue found — LOCO-B result applies unchanged


## Cell 6 — Supervised Classification & Calibration Analysis

Calibration measures whether the model's *predicted probabilities* match *actual event rates*. A model can have perfect AUC but terrible calibration. Both are required for clinical deployment.

In [14]:
# ============================================================
# CELL 6: Supervised Classification + Calibration Analysis
# ============================================================
log("Step 6: Supervised classification and calibration...")

# ── 6a: Train/test split (no leakage) ────────────────────────
X_idx = X_scaled.index
X_tr_idx, X_te_idx, y_tr, y_te = train_test_split(
    X_idx, y_binary.values, test_size=0.20,
    stratify=y_binary.values, random_state=RANDOM_SEED)

# PCA fitted on training set only
pca_sup = PCA(n_components=n_comp, svd_solver="full", random_state=RANDOM_SEED)
pca_sup.fit(X_scaled.loc[X_tr_idx].values)
X_tr_pca = pca_sup.transform(X_scaled.loc[X_tr_idx].values)
X_te_pca = pca_sup.transform(X_scaled.loc[X_te_idx].values)

# ── 6b: Train classifiers ────────────────────────────────────
def train_eval(clf, Xtr, ytr, Xte, yte, name):
    clf.fit(Xtr, ytr)
    proba = clf.predict_proba(Xte)[:,1]
    pred  = (proba > 0.5).astype(int)
    roc   = roc_auc_score(yte, proba)
    pr    = average_precision_score(yte, proba)
    acc   = accuracy_score(yte, pred)
    f1    = f1_score(yte, pred)
    rec   = recall_score(yte, pred)
    brier = brier_score_loss(yte, proba)
    cm    = confusion_matrix(yte, pred)
    print(f"  {name:22s}  ROC={roc:.4f}  PR={pr:.4f}  "
          f"Acc={acc:.4f}  F1={f1:.4f}  Rec={rec:.4f}  Brier={brier:.4f}")
    return dict(model=clf, proba=proba, pred=pred, name=name,
                roc=roc, pr=pr, acc=acc, f1=f1, rec=rec, brier=brier, cm=cm)

print("\n[HASEM] Binary supervised classification results:")
lr  = LogisticRegression(max_iter=10000, solver="saga",
                         class_weight="balanced", random_state=RANDOM_SEED)
rf  = RandomForestClassifier(n_estimators=200, n_jobs=-1,
                              class_weight="balanced", random_state=RANDOM_SEED)
res_lr = train_eval(lr,  X_tr_pca, y_tr, X_te_pca, y_te, "Logistic Regression")
res_rf = train_eval(rf,  X_tr_pca, y_tr, X_te_pca, y_te, "Random Forest")
res_xgb = None
if HAVE_XGB:
    xgb_clf = xgb.XGBClassifier(n_estimators=200, eval_metric="logloss",
                                  random_state=RANDOM_SEED, n_jobs=-1)
    res_xgb = train_eval(xgb_clf, X_tr_pca, y_tr, X_te_pca, y_te, "XGBoost")

# ── 6c: ROC and PR curves ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for res in [res_lr, res_rf, res_xgb]:
    if res is None: continue
    RocCurveDisplay.from_predictions(
        y_te, res["proba"], name=f"{res['name']} (AUC={res['roc']:.4f})", ax=axes[0])
    PrecisionRecallDisplay.from_predictions(
        y_te, res["proba"], name=f"{res['name']} (AP={res['pr']:.4f})",  ax=axes[1])
axes[0].set_title("ROC Curves"); axes[1].set_title("Precision–Recall Curves")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,"supervised","roc_pr_curves.png"), dpi=150)
plt.close()

# ── 6d: CALIBRATION CURVES ───────────────────────────────────
log("Step 6d: Generating calibration curves (reliability diagrams)...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
models_to_cal = [r for r in [res_lr, res_rf, res_xgb] if r is not None]

for ax, res in zip(axes, models_to_cal):
    # Platt-scale calibration wrapper
    cal_clf = CalibratedClassifierCV(res["model"], cv="prefit", method="sigmoid")
    cal_clf.fit(X_te_pca, y_te)
    cal_proba = cal_clf.predict_proba(X_te_pca)[:,1]

    # Calibration curve: bins actual event rates vs predicted probabilities
    frac_pos, mean_pred = calibration_curve(y_te, res["proba"], n_bins=10)
    frac_pos_cal, mean_pred_cal = calibration_curve(y_te, cal_proba, n_bins=10)

    brier_raw = brier_score_loss(y_te, res["proba"])
    brier_cal = brier_score_loss(y_te, cal_proba)

    ax.plot([0,1],[0,1],"k--", lw=1, label="Perfect calibration")
    ax.plot(mean_pred, frac_pos, "o-", color="#F44336",
            label=f"Uncalibrated (Brier={brier_raw:.4f})")
    ax.plot(mean_pred_cal, frac_pos_cal, "s-", color="#1F4E79",
            label=f"Platt-scaled  (Brier={brier_cal:.4f})")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Fraction of Positives")
    ax.set_title(f"Calibration — {res['name']}")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Reliability Diagrams: Raw vs Platt-Scaled Calibration", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,"calibration","calibration_curves.png"), dpi=150)
plt.close()
log("Calibration curves saved.")

# ── 6e: Save supervised performance summary ─────────────────
sup_results = [res_lr, res_rf]
if res_xgb: sup_results.append(res_xgb)
sup_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ["model","proba","pred","cm"]}
    for r in sup_results
])
sup_df.to_csv(os.path.join(OUTDIR,"supervised","supervised_performance.csv"), index=False)

# Save models
joblib.dump({r["name"]:r["model"] for r in sup_results},
            os.path.join(OUTDIR,"models","supervised_models.joblib"))
log("Step 6 complete.")


[HASEM] Step 6: Supervised classification and calibration...

[HASEM] Binary supervised classification results:
  Logistic Regression     ROC=0.9998  PR=0.9999  Acc=0.9950  F1=0.9957  Rec=0.9959  Brier=0.0036
  Random Forest           ROC=0.9992  PR=0.9993  Acc=0.9869  F1=0.9887  Rec=0.9973  Brier=0.0323
  XGBoost                 ROC=0.9996  PR=0.9997  Acc=0.9945  F1=0.9952  Rec=0.9968  Brier=0.0045
[HASEM] Step 6d: Generating calibration curves (reliability diagrams)...
[HASEM] Calibration curves saved.
[HASEM] Step 6 complete.


## Cell 7 — Multiclass Cancer Type Classification

In [15]:
# ============================================================
# CELL 7: Multiclass Cancer Type Classification
# ============================================================
log("Step 7: Multiclass supervised classification...")

mask_tumour = (y_binary == 1)
X_tumour    = X_scaled.loc[mask_tumour]
y_tumour    = y_detailed.loc[mask_tumour].fillna("Unknown")

# Filter rare classes
counts       = y_tumour.value_counts()
valid_cls    = counts[counts >= 5].index
X_tumour     = X_tumour.loc[y_tumour.isin(valid_cls)]
y_tumour     = y_tumour.loc[y_tumour.isin(valid_cls)]

le_multi     = LabelEncoder()
y_enc        = le_multi.fit_transform(y_tumour)
class_names  = le_multi.classes_
n_classes    = len(class_names)
log(f"Multiclass: {len(y_tumour):,} samples across {n_classes} cancer types")

X_mtr_idx, X_mte_idx, y_mtr, y_mte = train_test_split(
    X_tumour.index, y_enc, test_size=0.2,
    stratify=y_enc, random_state=RANDOM_SEED)

pca_multi = PCA(n_components=n_comp, svd_solver="full", random_state=RANDOM_SEED)
pca_multi.fit(X_tumour.loc[X_mtr_idx].values)
X_mtr_pca = pca_multi.transform(X_tumour.loc[X_mtr_idx].values)
X_mte_pca = pca_multi.transform(X_tumour.loc[X_mte_idx].values)

def train_multi(clf, Xtr, ytr, Xte, yte, name):
    clf.fit(Xtr, ytr)
    preds = clf.predict(Xte)
    acc = accuracy_score(yte, preds)
    f1  = f1_score(yte, preds, average="macro")
    try:
        proba = clf.predict_proba(Xte)
        roc = roc_auc_score(
            pd.get_dummies(yte).values, proba, average="macro", multi_class="ovr")
    except:
        roc = np.nan
    print(f"  {name:22s}  ACC={acc:.4f}  F1={f1:.4f}  ROC={roc:.4f}")
    report = classification_report(
        yte, preds, target_names=class_names, output_dict=True, zero_division=0)
    return dict(model=clf, name=name, acc=acc, f1=f1, roc=roc, report=report)

print("\n[HASEM] Multiclass results:")
lr_m  = train_multi(
    LogisticRegression(max_iter=5000, solver="saga",
                       multi_class="multinomial", n_jobs=-1, random_state=RANDOM_SEED),
    X_mtr_pca, y_mtr, X_mte_pca, y_mte, "Logistic Regression")
rf_m  = train_multi(
    RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_SEED),
    X_mtr_pca, y_mtr, X_mte_pca, y_mte, "Random Forest")
res_xgb_m = None
if HAVE_XGB:
    res_xgb_m = train_multi(
        xgb.XGBClassifier(n_estimators=300, objective="multi:softprob",
                           num_class=n_classes, eval_metric="mlogloss",
                           random_state=RANDOM_SEED, n_jobs=-1),
        X_mtr_pca, y_mtr, X_mte_pca, y_mte, "XGBoost")

# Save per-class classification reports
sup_multi_dir = os.path.join(OUTDIR,"supervised")
for res in [lr_m, rf_m, res_xgb_m]:
    if res is None: continue
    tag = res["name"].replace(" ","_")
    pd.DataFrame(res["report"]).T.to_csv(
        os.path.join(sup_multi_dir, f"{tag}_multiclass_report.csv"))

log("Step 7 complete.")


[HASEM] Step 7: Multiclass supervised classification...
[HASEM] Multiclass: 10,959 samples across 32 cancer types

[HASEM] Multiclass results:
  Logistic Regression     ACC=0.9767  F1=0.9447  ROC=0.9997
  Random Forest           ACC=0.9516  F1=0.8856  ROC=0.9982
  XGBoost                 ACC=0.9567  F1=0.9076  ROC=0.9977
[HASEM] Step 7 complete.


## Cell 8 — Gene Contribution Back-Mapping & Complete Pathway Enrichment Statistics

This cell produces the full pathway enrichment table with **p-values, adjusted p-values (FDR), overlap counts and precision** — the exact statistics reviewers require.

In [ ]:
# =======================```````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````==================================
# CELL 8 (FIXED): Gene Contribution Mapping & Pathway Enrichment
# Fixes: SHAP version, Ensembl version stripping, g:Profiler FDR
# ============================================================
log("Step 8: Gene contribution mapping and pathway enrichment...")
contr_dir = os.path.join(OUTDIR, "contributions")
path_dir  = os.path.join(OUTDIR, "pathway_enrichment")

gene_names = list(X_scaled.columns)

# ── 8a: Permutation importance (stable across SHAP versions) ──
log("8a: Computing permutation importance on binary Random Forest...")
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    res_rf["model"], X_te_pca, y_te,
    n_repeats=15,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
pc_importance = perm.importances_mean
log(f"Permutation importance computed for {len(pc_importance)} PCs.")

# ── 8b: Map PC importance → gene contributions ────────────────
log("8b: Back-projecting PC importance to gene space...")
abs_loadings = np.abs(pca_sup.components_.T)   # (n_genes × n_PCs)
pc_imp       = np.array(pc_importance[:abs_loadings.shape[1]])
gene_scores  = abs_loadings.dot(pc_imp)

gene_contrib = pd.DataFrame({
    "gene_raw":      gene_names,
    "contribution":  gene_scores
}).sort_values("contribution", ascending=False).reset_index(drop=True)
gene_contrib["rank"] = range(1, len(gene_contrib) + 1)

# ── 8c: Strip Ensembl version numbers before conversion ───────
# e.g. ENSG00000230910.2 → ENSG00000230910
def strip_version(eid):
    return str(eid).split(".")[0] if "." in str(eid) else str(eid)

gene_contrib["gene"] = gene_contrib["gene_raw"].apply(strip_version)
log(f"Example cleaned IDs: {gene_contrib['gene'].head(3).tolist()}")

# ── 8d: Ensembl → HGNC symbol conversion ─────────────────────
log("8c: Converting Ensembl IDs to HGNC symbols...")
try:
    import mygene
    mg  = mygene.MyGeneInfo()
    ensg_ids = gene_contrib["gene"].tolist()

    # Query in batches of 1000 to avoid timeouts
    all_results = []
    for i in range(0, len(ensg_ids), 1000):
        batch = ensg_ids[i:i+1000]
        res   = mg.querymany(batch, scopes="ensembl.gene",
                             fields="symbol", species="human",
                             verbose=False)
        all_results.extend(res)

    conv_map = {}
    for r in all_results:
        if "symbol" in r:
            conv_map[r["query"]] = r["symbol"]
        else:
            conv_map[r["query"]] = r["query"]   # fallback to Ensembl ID

    gene_contrib["gene_hgnc"] = gene_contrib["gene"].map(conv_map).fillna(gene_contrib["gene"])
    n_converted = gene_contrib["gene_hgnc"].apply(
        lambda x: not x.startswith("ENSG")).sum()
    log(f"Converted {n_converted:,} of {len(gene_contrib):,} genes to HGNC symbols.")

except Exception as e:
    log(f"Gene conversion error: {e} — using Ensembl IDs directly.")
    gene_contrib["gene_hgnc"] = gene_contrib["gene"]

# Save full contribution table
gene_contrib.to_csv(
    os.path.join(contr_dir, "gene_contributions_global.csv"), index=False)
log(f"Top 10 genes: {gene_contrib['gene_hgnc'].head(10).tolist()}")

# ── 8e: g:Profiler global enrichment (top 200 genes) ─────────
log("8d: Running g:Profiler global enrichment (top 200 genes)...")
from gprofiler import GProfiler
gp = GProfiler(return_dataframe=True)

top200 = gene_contrib.head(200)["gene_hgnc"].dropna().tolist()
# Remove any remaining Ensembl IDs if conversion failed for some
top200_clean = [g for g in top200 if g and len(g) > 1]

log(f"Submitting {len(top200_clean)} genes to g:Profiler...")
try:
    enrich_global = gp.profile(
        organism  = "hsapiens",
        query     = top200_clean,
        sources   = ["GO:BP", "GO:MF", "KEGG", "REAC"],
        user_threshold = 0.05,
        significance_threshold_method = "fdr"
    )

    if enrich_global.empty:
        log("WARNING: g:Profiler returned no significant terms.")
    else:
        # Clean up columns for manuscript
        rename_map = {
            "source":            "Database",
            "name":              "Pathway/Term",
            "native":            "Term ID",
            "p_value":           "p-value",
            "intersection_size": "Overlap N",
            "term_size":         "Term Size",
            "precision":         "Precision",
            "recall":            "Recall"
        }
        enrich_global.rename(columns=rename_map, inplace=True, errors="ignore")

        # Compute FDR safely
        if "p-value" in enrich_global.columns:
            pvals = enrich_global["p-value"].fillna(1.0).clip(0, 1).values
            from scipy.stats import false_discovery_control
            try:
                enrich_global["p-adj (FDR)"] = false_discovery_control(pvals)
            except Exception:
                # Manual Benjamini-Hochberg fallback
                n = len(pvals)
                order  = np.argsort(pvals)
                ranks  = np.empty(n); ranks[order] = np.arange(1, n+1)
                fdr    = np.minimum(1, pvals * n / ranks)
                enrich_global["p-adj (FDR)"] = fdr

        enrich_global = enrich_global.sort_values("p-value")
        enrich_global.to_csv(
            os.path.join(path_dir, "gprofiler_global_top200_FULL.csv"), index=False)

        log(f"Enrichment complete: {len(enrich_global)} significant terms found.")
        print("\n[HASEM] Top 15 enriched pathways:")
        disp_cols = ["Database","Pathway/Term","Overlap N",
                     "p-value","p-adj (FDR)","Precision"]
        disp_cols = [c for c in disp_cols if c in enrich_global.columns]
        print(enrich_global.head(15)[disp_cols].to_string(index=False))

except Exception as e:
    log(f"g:Profiler enrichment error: {e}")
    enrich_global = pd.DataFrame()

# ── 8f: Per-cancer enrichment (top 3 cancer types by size) ───
log("8e: Running per-cancer enrichment for largest tumour types...")
top_cancers = (y_detailed[y_binary==1]
               .value_counts().head(3).index.tolist())

for cancer in top_cancers:
    try:
        mask_c  = (y_detailed == cancer).values
        Xc      = X_scaled.values[mask_c]
        Xc_pca  = pca_sup.transform(Xc)

        # Use RF feature importances as a proxy for this cancer type
        perm_c  = permutation_importance(
            res_rf["model"], Xc_pca[:min(200, len(Xc_pca))],
            y_binary.values[mask_c][:min(200, len(Xc_pca))],
            n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1)

        gc = abs_loadings.dot(perm_c.importances_mean[:abs_loadings.shape[1]])
        gdf = pd.DataFrame({"gene":gene_names,"contribution":gc})
        gdf = gdf.sort_values("contribution", ascending=False)

        top_g = gdf.head(200)["gene"].apply(strip_version).tolist()
        top_g_hgnc = [conv_map.get(g, g) for g in top_g]
        top_g_clean = [g for g in top_g_hgnc if g and len(g) > 1]

        r = gp.profile(
            organism  = "hsapiens",
            query     = top_g_clean,
            sources   = ["GO:BP", "KEGG", "REAC"],
            user_threshold = 0.05,
            significance_threshold_method = "fdr"
        )
        tag = cancer.replace("/","_").replace(" ","_")
        r.to_csv(os.path.join(path_dir,
                 f"gprofiler_{tag}_top200.csv"), index=False)
        log(f"  {cancer}: {len(r)} enriched terms saved.")
    except Exception as ex:
        log(f"  {cancer}: enrichment failed ({ex})")

log("Step 8 complete.")

[HASEM] Step 8: Gene contribution mapping and pathway enrichment...
[HASEM] 8a: Computing permutation importance on binary Random Forest...


## Cell 9 — Final Summary Report

Prints a clean summary of all results, ready to copy into the manuscript.

In [ ]:
# ============================================================
# CELL 9: Final Summary Report — all numbers for manuscript
# ============================================================
print("=" * 65)
print("  HASEM — COMPLETE RESULTS SUMMARY")
print("=" * 65)

print("\n── DATASET ──────────────────────────────────────────────")
print(f"  Total samples       : {len(y_binary):,}")
print(f"  Healthy (GTEx)      : {(y_binary==0).sum():,}")
print(f"  Tumour (TCGA/TARGET): {(y_binary==1).sum():,}")
print(f"  Genes scanned       : {len(mad_df):,}")
print(f"  Genes retained (MAD): {TOP_K:,}")
print(f"  PCA components      : {n_comp}")

print("\n── BATCH-EFFECT ANALYSIS ────────────────────────────────")
print(f"  Cohort separability ROC-AUC: {batch_auc:.4f}")
print(f"  Interpretation: {'Low batch effect' if batch_auc < 0.80 else 'Batch signal present — report in paper'}")

print("\n── UNSUPERVISED DETECTION (T²+Q vs Isolation Forest) ────")
for r in [res_emp, res_theo, res_iso]:
    print(f"  {r['label']:30s}  ROC={r['roc']:.4f}  PR={r['pr']:.4f}  "
          f"Det={r['detect']:.2f}%  FA={r['false_alarm']:.2f}%")

print("\n── PER-CANCER DETECTION (top 10, LOCO-A) ───────────────")
print(pc_emp.head(10)[["Cancer Type","N Samples","Detection Rate (%)","95% CI Lower","95% CI Upper"]].to_string(index=False))

print("\n── PER-CANCER DETECTION (top 10, LOCO-B) ───────────────")
print(df_locoB.head(10).to_string(index=False))

print("\n── SUPERVISED CLASSIFICATION ────────────────────────────")
for res in [res_lr, res_rf, res_xgb]:
    if res is None: continue
    print(f"  {res['name']:22s}  ROC={res['roc']:.4f}  PR={res['pr']:.4f}  "
          f"Acc={res['acc']:.4f}  F1={res['f1']:.4f}  Brier={res['brier']:.4f}")

print("\n── PATHWAY ENRICHMENT (top 10 global) ──────────────────")
if not enrich_global.empty:
    cols = ["Database","Pathway/Term","Overlap N","p-value","p-adj (FDR)","Precision"]
    cols = [c for c in cols if c in enrich_global.columns]
    print(enrich_global.head(10)[cols].to_string(index=False))
else:
    print("  (g:Profiler results — see pathway_enrichment/ directory)")

print("\n── OUTPUT FILES ─────────────────────────────────────────")
print(f"  results_hasem/")
print(f"  ├── batch_effects/         pca_biplot, batch_summary.csv")
print(f"  ├── unsupervised/          T2_Q_scatter, unsup_metrics.csv")
print(f"  ├── per_cancer/            per_cancer_empirical.csv (with CIs)")
print(f"  ├── loco/                  LOCO_A_results.csv, LOCO_B_results.csv")
print(f"  ├── calibration/           calibration_curves.png")
print(f"  ├── supervised/            roc_pr_curves.png, performance.csv")
print(f"  ├── pathway_enrichment/    gprofiler_global_top200_FULL.csv")
print(f"  ├── contributions/         gene_contributions_global.csv")
print(f"  └── models/                all .joblib artefacts")
print("=" * 65)
print("  All numbers above are ready to copy into your manuscript.")
print("=" * 65)
